# Credit Card Fraud Detection

End-to-end machine learning pipeline to detect fraudulent credit card transactions.

**Dataset:** [Credit Card Fraud Detection — Kaggle](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)
Transactions made by European cardholders in September 2013. It contains 284,807 transactions, of which only 492 are fraudulent (≈ 0.17%), making this a highly **imbalanced** classification problem.

**Features:**
- `Time`, `Amount` — original features
- `V1` … `V28` — anonymized features obtained from a PCA transformation (the original ones are not provided for confidentiality)
- `Class` — target variable (0 = normal transaction, 1 = fraud)

**Pipeline:**
1. Load & inspect the data
2. Preprocess (scaling, drop unused columns, remove duplicates)
3. Train baseline models on the imbalanced data
4. Handle class imbalance with **Undersampling**
5. Handle class imbalance with **Oversampling (SMOTE)**
6. Persist the final model with `joblib`
7. Run an example inference

## 1. Setup and Imports

In [ ]:
# Core data-handling library
import pandas as pd

## 2. Load the Dataset

The CSV file is expected to be in the same folder as this notebook. You can download it from the Kaggle link above.

In [ ]:
# Read the dataset into a pandas DataFrame
df_credit_card = pd.read_csv("creditcard.csv")

In [ ]:
# Preview the first five rows
df_credit_card.head()

In [ ]:
# By default pandas hides middle columns when there are many.
# This option forces pandas to display all 31 columns so we can inspect everything.
pd.options.display.max_columns = None

df_credit_card.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Dataset dimensions: (rows, columns)
df_credit_card.shape

In [ ]:
# Column data types and non-null counts
df_credit_card.info()

In [ ]:
# Check for missing values column by column
df_credit_card.isnull().sum()

## 4. Data Preprocessing

Three steps:
1. **Scale the `Amount` column** — every other feature (`V1`…`V28`) is already PCA-transformed and roughly centred around zero, but `Amount` is on a much larger scale. Standardising it puts it on the same footing.
2. **Drop the `Time` column** — it represents seconds elapsed since the first transaction and is not directly informative for classification.
3. **Remove duplicate rows** — duplicates would bias both training and evaluation.

In [ ]:
# StandardScaler will transform Amount to have mean=0 and std=1
from sklearn.preprocessing import StandardScaler

# Silence non-critical warnings for cleaner output
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Fit the scaler on the Amount column and replace it with the scaled values
sc = StandardScaler()
df_credit_card['Amount'] = sc.fit_transform(
    pd.DataFrame(df_credit_card['Amount']))

df_credit_card.head()

In [ ]:
# Drop the Time column — it adds little predictive value for this task
df_credit_card = df_credit_card.drop(['Time'], axis=1)
df_credit_card.head()

In [ ]:
# Check whether any duplicate rows exist
df_credit_card.duplicated().any()

In [ ]:
# Remove duplicate rows and verify the new shape
df_credit_card = df_credit_card.drop_duplicates()
df_credit_card.shape

## 5. Class Imbalance Investigation

Before training, let's confirm just how imbalanced the target variable is. This will guide the modelling strategy.

In [ ]:
# Count how many transactions belong to each class
# 0 = normal, 1 = fraud
df_credit_card['Class'].value_counts()

In [ ]:
# Visual confirmation of the imbalance
import seaborn as sns
import matplotlib.pyplot as plt

plt.style.use('ggplot')

In [ ]:
# Count plot of the target classes
sns.countplot(x='Class', data=df_credit_card)
plt.title('Transaction Class Distribution (0 = Normal, 1 = Fraud)')
plt.show()

## 6. Baseline Models on Imbalanced Data

Train two classifiers directly on the imbalanced data to establish a baseline.
Because the dataset is so skewed, **accuracy is a misleading metric here** — a model that always predicts "normal" would already score above 99%. We therefore also report **precision**, **recall** and **F1**.

In [ ]:
# Separate features (X) from target (y)
X = df_credit_card.drop('Class', axis=1)
y = df_credit_card['Class']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# 80/20 train/test split, fixed random_state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [ ]:
# Models and evaluation metrics
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [ ]:
# Train and evaluate both classifiers on the imbalanced dataset
classifier = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree Classifier": DecisionTreeClassifier(random_state=42)
}

for name, clf in classifier.items():
    print(f'\n =========={name}=========')
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(f'\n Accuracy: {accuracy_score(y_test, y_pred)}')
    print(f'\n Precision: {precision_score(y_test, y_pred)}')
    print(f'\n Recall: {recall_score(y_test, y_pred)}')
    print(f'\n F1 Score: {f1_score(y_test, y_pred)}')

## 7. Handling Imbalance — Undersampling

**Idea:** randomly drop most "normal" transactions so the two classes match in size.
**Pro:** fast, simple. **Con:** we throw away a huge amount of information.

In [ ]:
# Split the dataset by class
normal = df_credit_card[df_credit_card['Class'] == 0]
fraud  = df_credit_card[df_credit_card['Class'] == 1]

In [ ]:
# Randomly sample the same number of normal transactions as the number of frauds
# (473 = number of fraud rows remaining after deduplication)
normal_sample = normal.sample(n=473, random_state=42)
normal_sample.shape

In [ ]:
# Concatenate the two balanced subsets into a single DataFrame
df_under_samplig = pd.concat([normal_sample, fraud], ignore_index=True)
df_under_samplig.head()

In [ ]:
# Confirm both classes now have the same count
df_under_samplig['Class'].value_counts()

In [ ]:
# Rebuild X and y from the balanced (undersampled) DataFrame
X = df_under_samplig.drop('Class', axis=1)
y = df_under_samplig['Class']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [ ]:
# Train and evaluate the same two classifiers, now on balanced data
classifier = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree Classifier": DecisionTreeClassifier(random_state=42)
}

for name, clf in classifier.items():
    print(f'\n =========={name}=========')
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(f'\n Accuracy: {accuracy_score(y_test, y_pred)}')
    print(f'\n Precision: {precision_score(y_test, y_pred)}')
    print(f'\n Recall: {recall_score(y_test, y_pred)}')
    print(f'\n F1 Score: {f1_score(y_test, y_pred)}')

## 8. Handling Imbalance — Oversampling with SMOTE

**Idea:** instead of throwing data away, *synthesise* new minority-class samples.
**SMOTE** (Synthetic Minority Over-sampling TEchnique) creates synthetic fraud points by interpolating between existing ones.
**Pro:** keeps all the original information. **Con:** can create unrealistic samples if the minority class is too small or noisy.

In [ ]:
# Rebuild X and y from the full (deduplicated) DataFrame
X = df_credit_card.drop('Class', axis=1)
y = df_credit_card['Class']

In [ ]:
X.shape

In [ ]:
y.shape

In [ ]:
# imbalanced-learn provides SMOTE
from imblearn.over_sampling import SMOTE

In [ ]:
# Generate synthetic fraud samples so both classes have the same size
X_res, y_res = SMOTE(random_state=42).fit_resample(X, y)

In [ ]:
# Confirm the new balance after SMOTE
y_res.value_counts()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42)

In [ ]:
# Train and evaluate both classifiers on the SMOTE-balanced data
classifier = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree Classifier": DecisionTreeClassifier(random_state=42)
}

for name, clf in classifier.items():
    print(f'\n =========={name}=========')
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(f'\n Accuracy: {accuracy_score(y_test, y_pred)}')
    print(f'\n Precision: {precision_score(y_test, y_pred)}')
    print(f'\n Recall: {recall_score(y_test, y_pred)}')
    print(f'\n F1 Score: {f1_score(y_test, y_pred)}')

## 9. Save the Final Model

The Decision Tree trained on the SMOTE-resampled data gave the best results, so we retrain it on the **full** resampled dataset and persist it with `joblib` for later reuse.

In [ ]:
# Retrain the chosen model on the full resampled dataset
dtc = DecisionTreeClassifier(random_state=42)
dtc.fit(X_res, y_res)

In [ ]:
# joblib is the standard sklearn-friendly serialization library
import joblib

In [ ]:
# Persist the trained model to disk
joblib.dump(dtc, "Credit_Card_Model.pkl")

## 10. Inference Example

Load the persisted model and run a prediction on a single transaction.
The feature vector below corresponds to the very first row of the original dataset (a known *normal* transaction).

In [ ]:
# Reload the model from disk — exactly how it would be used in production
model = joblib.load("Credit_Card_Model.pkl")

In [ ]:
# Predict on a single transaction (features must be in the same order as during training)
predict = model.predict([[-1.3598071336738, -0.0727811733098497, 2.53634673796914, 1.37815522427443, -0.338320769942518, 0.462387777762292, 0.239598554061257, 0.0986979012610507, 0.363786969611213, 0.0907941719789316, -0.551599533260813, -0.617800855762348, -0.991389847235408, -0.311169353699879,
                        1.46817697209427, -0.470400525259478, 0.207971241929242, 0.0257905801985591, 0.403992960255733, 0.251412098239705, -0.018306777944153, 0.277837575558899, -0.110473910188767, 0.0669280749146731, 0.128539358273528, -0.189114843888824, 0.133558376740387, -0.0210530534538215, 149.62]])

In [ ]:
# Raw prediction (0 or 1)
predict[0]

In [ ]:
# Human-readable interpretation
if predict[0] == 0:
    print("Normal Transaction")
else:
    print("Fraud Transaction")

## 11. Conclusions & Next Steps

- The **imbalanced baseline** achieves very high accuracy but its recall on the fraud class is poor — it misses real fraud cases.
- **Undersampling** dramatically improves recall but is trained on very few samples, so it generalises less.
- **SMOTE oversampling** gives the best overall trade-off in this notebook and is the strategy used for the final model.

**Possible improvements:**
- Try ensemble methods such as `RandomForestClassifier` or `XGBoost` on the SMOTE data.
- Use **stratified k-fold cross-validation** instead of a single train/test split.
- Add the **Precision–Recall AUC** metric — it is more informative than ROC-AUC for highly imbalanced problems.
- Tune hyper-parameters with `GridSearchCV` or `RandomizedSearchCV`.
- Evaluate the final model on the **original imbalanced test set** (not the SMOTE-resampled one) to get a realistic performance estimate.